# Question-Driven EDA : Ames Housing Data

[Contents]  
- 다양한 시각화와 통계 정보를 활용하여 데이터의 특성을 이해하고, 전처리가 필요한 문제점을 직접 발견하는 것이 목표
- 각 단계에서 얻은 결과를 바탕으로 왜 이러한 현상이 나타나는지, 어떤 전처리가 필요할지 스스로 판단해 보는 것이 이번 실습의 방향성  
- EDA 결과를 정답처럼 받아들이기보다, 관찰한 근거를 바탕으로 질문에 답하는 것 권장  
  
[Data]  
Ames Housing Raw 데이터를 바탕으로 교육용으로 간소화
- Target: SalePrice
- Numeric Features: 10개
- Categorical Features: 4개

[Note]  
- 배포된 "ames_housing_edu.csv" 파일을 직접 업로드


## 1. 라이브러리 Import

- 데이터 처리: pandas, numpy
- 시각화: matplotlib, seaborn


In [ ]:
import io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("pandas version:", pd.__version__)

## 2. Config

- 실수값을 소수 둘째 자리까지 출력


In [ ]:
pd.set_option("display.float_format", "{:.2f}".format)

## 3. 데이터 업로드

- 배포된 "ames_housing_edu.csv" 파일을 업로드
- 업로드한 파일을 DataFrame으로 로드


In [ ]:
# Colab에서는 업로드 창을 사용하고, 로컬 실행에서는 같은 폴더의 CSV를 읽습니다.
csv_filename = "ames_housing_edu.csv"

try:
    from google.colab import files

    uploaded = files.upload()
    if csv_filename not in uploaded:
        raise FileNotFoundError(
            f"{csv_filename} 파일을 업로드 하세요. "
            f"업로드된 파일: {list(uploaded.keys())}"
        )
    df = pd.read_csv(io.BytesIO(uploaded[csv_filename]))
except ImportError:
    local_csv_path = Path(csv_filename)
    if not local_csv_path.exists():
        local_csv_path = Path.cwd() / csv_filename
    if not local_csv_path.exists():
        raise FileNotFoundError(
            f"{csv_filename}을 노트북과 같은 폴더에 두고 다시 실행하세요. "
            f"현재 작업 폴더: {Path.cwd()}"
        )
    df = pd.read_csv(local_csv_path)

print("Loaded data shape:", df.shape)
display(df.head())

## 4. 데이터 구조 확인

[Question 1]  
- 데이터는 몇 개의 행과 열로 구성되어 있는가?
- Target 변수는 무엇인가?
- 수치형 변수와 범주형 변수는 각각 몇 개인가?
- 분석 전에 자료형을 변경해야 할 변수는 보이는가?


### 답변

- **데이터 크기:** 2,930행 × 15열이다. 관측 단위는 주택 매매 1건이다.
- **Target:** `SalePrice`이다.
- **변수 구성:** 수치형 열은 11개이며, 이 중 `SalePrice`를 제외하면 수치형 설명변수는 10개이다. 범주형 설명변수는 4개(`Neighborhood`, `House_Style`, `Kitchen_Qual`, `Exter_Qual`)이다.
- **자료형 재검토:** `Year_Built`, `Garage_Yr_Blt`는 숫자로 저장되어 있지만 ‘연도’라는 의미를 가진다. 그대로 크기 변수로 사용하기보다 매매 시점이 있다면 주택/차고 연식으로 변환하거나 구간화하는 편이 해석에 유리하다. `Overall_Qual`, `Overall_Cond`, `Kitchen_Qual`, `Exter_Qual`은 순서가 있는 등급 변수이므로 명목형 변수와 구분해야 한다.


In [ ]:
print("Data shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nData samples:")
display(df.head())


In [ ]:
# 수치형 변수와 범주형 변수 구분
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(include="object").columns.tolist()

print("Numeric variables:", len(numeric_columns), numeric_columns)
print("Categorical variables:", len(categorical_columns), categorical_columns)

## 5. 기술통계 확인

[Question 2]  
- 주택가격의 평균과 중앙값은 비슷한가?
- 평균과 중앙값의 차이는 분포의 어떤 특징을 암시하는가?
- 최소값과 최대값의 차이가 큰 변수는 무엇인가?
- 변수의 단위가 서로 다르다는 점은 이후 분석에 어떤 영향을 줄 수 있는가?


### 답변

- `SalePrice` 평균은 **$180,796.06**, 중앙값은 **$160,000**으로 약 **$20,796** 차이가 난다. 평균이 더 큰 것은 일부 고가 주택이 분포의 오른쪽 꼬리를 늘렸음을 암시한다.
- 범위가 특히 큰 변수는 `SalePrice`($12,789~$755,000), `Lot_Area`(1,300~215,245 ft²), `Gr_Liv_Area`(334~5,642 ft²), `Total_Bsmt_SF`(0~6,110 ft²)이다.
- 변수마다 단위와 범위가 크게 다르므로 거리 기반 모델(KNN 등)이나 규제가 있는 선형모델에서는 큰 단위의 변수가 학습을 지배할 수 있다. 학습 데이터에서 `StandardScaler`/`RobustScaler`를 적합하고, 강하게 치우친 면적 변수는 로그 변환을 함께 검토한다. 트리 모델은 스케일에 덜 민감하지만 결측·오류·왜도 확인은 여전히 필요하다.


In [ ]:
# 수치형 변수 기술통계
display(df[numeric_columns].describe().T)

# 범주형 변수 기술통계
display(df[categorical_columns].describe().T)

## 6. 결측치 확인

[Question 3]  
- 결측치가 있는 변수는 무엇인가?
- 결측치는 단순한 누락일까, 특정 시설이 없다는 의미일까?
- 결측치 비율이 분석에 영향을 줄 만큼 큰가?
- 결측치 처리 전에 원래 의미를 확인해야 하는 이유는 무엇인가?


### 답변

- 결측 변수는 `Garage_Yr_Blt` **159건(5.43%)**, `Total_Bsmt_SF` **1건(0.034%)**, `Garage_Cars` **1건(0.034%)**이다.
- `Garage_Yr_Blt` 결측 159건 중 **157건은 `Garage_Cars=0`**이므로 대부분 단순 누락이 아니라 **차고가 없음**을 뜻하는 구조적 결측으로 보인다. 나머지 2건(`Garage_Cars=1` 1건, `Garage_Cars` 자체 결측 1건)은 실제 누락 또는 입력 오류 후보이다.
- 5.43%는 행 전체를 삭제하기에는 작지 않다. `Has_Garage` 변수를 만들고, 차고가 없는 경우와 진짜 누락을 분리하는 방식이 적절하다.
- 결측의 원래 의미를 확인하지 않고 평균/중앙값으로 일괄 대치하면 ‘시설 없음’이라는 중요한 정보를 지우고, 존재하지 않는 차고 건축연도를 만들어낼 수 있다. `Total_Bsmt_SF`와 `Garage_Cars` 각 1건도 관련 변수와 원자료를 확인한 뒤 0 또는 조건부 중앙값 대치를 결정해야 한다.


In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Ratio(%)": df.isna().mean() * 100
}).sort_values("Missing_Count", ascending=False)

missing_summary = missing_summary[missing_summary["Missing_Count"] > 0]
display(missing_summary)

## 7. Target 분포 확인

[Question 4]  
- SalePrice는 대칭적인 분포인가?
- 고가 주택이 분포의 오른쪽 꼬리를 길게 만드는가?
- 평균과 중앙값 중 대표값으로 어느 값이 더 적절해 보이는가?
- 향후 회귀분석에서 Target 변환을 고려할 필요가 있는가?


### 답변

- `SalePrice`는 대칭이 아니며 **오른쪽 왜도 1.74**의 긴 오른쪽 꼬리를 가진다. 고가 주택이 평균을 중앙값보다 약 $20.8k 높게 만든다.
- 일반적인 주택의 대표값으로는 극단값 영향을 덜 받는 **중앙값 $160,000**이 평균보다 적절하다. 다만 총매출 규모를 설명하는 목적이라면 평균도 의미가 있으므로 목적에 따라 함께 제시한다.
- `log1p(SalePrice)`의 왜도는 약 **-0.01**로 거의 대칭이 된다. 따라서 회귀에서는 로그 타깃을 후보로 두고 잔차 안정화 여부와 검증 성능을 비교할 가치가 있다.
- 로그 타깃 모델을 사용할 때는 예측을 원 단위로 되돌린 뒤 MAE/RMSE를 함께 보고, 단순 지수 역변환에서 생길 수 있는 재변환 편향도 확인한다.


In [ ]:
target = "SalePrice"

print("Mean   :", df[target].mean())
print("Median :", df[target].median())
print("Skewness:", df[target].skew())

plt.figure(figsize=(10, 5))
sns.histplot(data=df, x=target, bins=40, kde=True)
plt.axvline(df[target].mean(), linestyle="--", label="Mean")
plt.axvline(df[target].median(), linestyle=":", label="Median")
plt.title("Distribution of SalePrice")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 3))
sns.boxplot(data=df, x=target)
plt.title("Box Plot of SalePrice")
plt.tight_layout()
plt.show()

## 8. 수치형 변수 분포 확인

[Question 5]  
- 강한 오른쪽 왜도를 보이는 변수는 무엇인가?
- 값이 특정 구간에 집중된 변수는 무엇인가?
- 이상치 후보가 많은 변수는 무엇인가?
- 모든 수치형 변수에 동일한 전처리를 적용하는 것이 적절한가?


### 답변

- 강한 오른쪽 왜도는 `Lot_Area` **12.82**, `Gr_Liv_Area` **1.27**, `Total_Bsmt_SF` **1.16**에서 나타난다. Target인 `SalePrice`도 1.74이다.
- `Overall_Cond`는 중앙값 5, IQR 5~6에 집중되고, `Overall_Qual`은 5~7, `Garage_Cars`와 `Full_Bath`는 주로 1~2, `Bedroom_AbvGr`는 2~3에 집중된다. 이 변수들은 연속형이라기보다 작은 정수 범위의 등급/개수 변수에 가깝다.
- IQR 기준 이상치 후보가 많은 설명변수는 `Overall_Cond` 252건, `Lot_Area` 127건, `Total_Bsmt_SF` 123건, `Bedroom_AbvGr` 78건, `Gr_Liv_Area` 75건이다.
- 모든 수치형 변수에 동일한 전처리를 적용하는 것은 적절하지 않다. 면적·가격은 로그/Robust scaling 후보이고, 연도는 연식 파생변수, 품질·상태는 순서형 변수, 0이 ‘시설 없음’을 뜻하는 지하실/차고 변수는 존재 여부 indicator와 함께 처리하는 편이 낫다.


In [ ]:
# 수치형 설명변수 분포
numeric_features = [column for column in numeric_columns if column != target]

df[numeric_features].hist(
    bins=30,
    figsize=(16, 12),
    edgecolor="black"
)
plt.suptitle("Distribution of Numeric Features", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
skewness = (
    df[numeric_features]
    .skew(numeric_only=True)
    .sort_values(ascending=False)
    .to_frame("Skewness")
)

display(skewness)

## 9. 이상치 후보 확인

[Question 6]  
- IQR 기준으로 이상치 후보가 많은 변수는 무엇인가?
- 통계적 이상치가 반드시 오류 데이터라고 볼 수 있는가?
- Gr_Liv_Area, Lot_Area, SalePrice의 극단값은 실제 대형 주택일 가능성이 있는가?


### 답변

- IQR 후보 수는 `Overall_Cond` **252건(8.60%)**, `SalePrice` **137건(4.68%)**, `Lot_Area` **127건(4.33%)**, `Total_Bsmt_SF` **123건(4.20%)**, `Bedroom_AbvGr` **78건(2.66%)**, `Gr_Liv_Area` **75건(2.56%)** 순으로 많다.
- 통계적 이상치는 해당 분포에서 드물다는 뜻이지 오류라는 뜻은 아니다. 특히 `Overall_Cond`는 1~9의 유효 등급이므로 IQR 경계 밖이라는 이유만으로 삭제하면 실제 상태가 매우 좋거나 나쁜 주택을 잃는다.
- `Gr_Liv_Area`, `Lot_Area`, `SalePrice`의 극단값은 실제 대형 부지·대형 주택·고급 주택일 수 있으므로 주소/거래 상태 등 원자료 확인이 먼저다. 다만 생활면적 4,000 ft² 초과 5건 중 Edwards의 3건은 품질 10임에도 가격이 $160k~$185k로 낮아 영향점 후보이다. 포함/제외 민감도 분석을 권장한다.
- 별도로 `Garage_Yr_Blt=2207` 1건은 데이터 기간(최대 2010년)을 벗어나는 명백한 입력 오류 후보이므로 수정하거나 결측 처리해야 한다.


In [ ]:
def iqr_outlier_summary(data, columns):
    results = []

    for column in columns:
        series = data[column].dropna()
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_count = ((series < lower_bound) | (series > upper_bound)).sum()

        results.append({
            "Variable": column,
            "Lower_Bound": lower_bound,
            "Upper_Bound": upper_bound,
            "Outlier_Count": outlier_count,
            "Outlier_Ratio(%)": outlier_count / len(series) * 100
        })

    return pd.DataFrame(results).sort_values("Outlier_Count", ascending=False)

outlier_summary = iqr_outlier_summary(df, numeric_columns)
display(outlier_summary)

In [ ]:
boxplot_columns = ["SalePrice", "Gr_Liv_Area", "Lot_Area", "Total_Bsmt_SF"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for axis, column in zip(axes, boxplot_columns):
    sns.boxplot(data=df, x=column, ax=axis)
    axis.set_title(f"Box Plot of {column}")

plt.tight_layout()
plt.show()

## 10. 범주형 변수 분포 확인

[Question 7]  
- 범주 수가 가장 많은 변수는 무엇인가?
- 특정 범주에 관측값이 지나치게 집중되어 있는가?
- 표본 수가 매우 적은 범주는 향후 인코딩과 모델링에 어떤 문제를 만들 수 있는가?


### 답변

- 범주 수가 가장 많은 변수는 `Neighborhood`로 **28개 범주**이다. `House_Style`은 8개, `Kitchen_Qual`은 5개, `Exter_Qual`은 4개이다.
- 관측치가 집중된 범주는 `House_Style=1Story` **50.55%**, `Kitchen_Qual=TA` **50.99%**, `Exter_Qual=TA` **61.40%**이다. `Neighborhood`의 최빈 범주는 `NAmes` 443건(15.12%)이다.
- 매우 희소한 범주는 `Neighborhood=Landmrk` 1건, `GrnHill` 2건, `Kitchen_Qual=Po` 1건 등이다. 이런 범주는 범주별 가격 추정치의 분산이 크고, 원-핫 인코딩 시 거의 0인 열을 만들며, 데이터 분할 후 학습 세트에 없는 범주가 검증/테스트에 나타날 수 있다.
- 의미가 유사한 희소 범주는 학습 데이터 기준으로 `Other`에 통합하거나, 규제가 있는 인코딩을 사용한다. 통합 기준과 인코더는 반드시 학습 세트에서만 결정해 누수를 막는다.


In [ ]:
for column in categorical_columns:
    print(f"\n=== {column} ===")
    display(
        df[column]
        .value_counts(dropna=False)
        .to_frame("Count")
        .assign(Ratio=lambda x: x["Count"] / len(df) * 100)
    )


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for axis, column in zip(axes, categorical_columns):
    order = df[column].value_counts().index
    sns.countplot(data=df, y=column, order=order, ax=axis)
    axis.set_title(f"Count of {column}")
    axis.set_xlabel("Count")

plt.tight_layout()
plt.show()

## 11. 수치형 변수와 주택가격의 관계

[Question 8]  
- 주택가격과 가장 강한 양의 관계를 보이는 변수는 무엇인가?
- 품질 점수와 면적 중 어느 변수가 가격 차이를 더 분명하게 설명하는가?
- 상관계수가 낮아도 비선형 관계가 존재할 수 있는가?
- 상관관계만으로 인과관계를 주장할 수 있는가?


### 답변

- `SalePrice`와 가장 강한 양의 Pearson 상관을 보이는 변수는 `Overall_Qual` **0.799**이다. 다음은 `Gr_Liv_Area` **0.707**, `Garage_Cars` **0.648**, `Total_Bsmt_SF` **0.632**이다.
- 품질 점수는 1에서 10으로 높아질수록 가격 중앙값이 약 **$50,150 → $451,950**으로 단계적으로 상승해 가격대를 더 선명하게 구분한다. 생활면적도 강한 양의 관계가 있지만 같은 면적에서도 품질과 지역에 따라 가격 분산이 크다.
- 낮은 Pearson 상관이 관계 부재를 뜻하지는 않는다. 예를 들어 `Lot_Area`는 Pearson 0.267이지만 Spearman 상관은 약 0.429로, 극단값과 비선형·단조 관계의 영향을 시사한다. 산점도, 구간별 중앙값, 변환 후 관계를 함께 확인해야 한다.
- 상관관계는 인과관계가 아니다. 품질·면적·지역·건축연도 등이 서로 연관되어 있고 관측자료에는 교란변수가 있으므로, 상관만으로 ‘품질 향상이 가격을 이만큼 올린다’고 주장할 수 없다.


In [ ]:
correlation = (
    df[numeric_columns]
    .corr(numeric_only=True)[target]
    .sort_values(ascending=False)
    .to_frame("Correlation_with_SalePrice")
)

display(correlation)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    df[numeric_columns].corr(numeric_only=True),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
relationship_columns = [
    "Overall_Qual",
    "Gr_Liv_Area",
    "Year_Built",
    "Total_Bsmt_SF"
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for axis, column in zip(axes, relationship_columns):
    sns.scatterplot(data=df, x=column, y=target, alpha=0.6, ax=axis)
    sns.regplot(data=df, x=column, y=target, scatter=False, ax=axis)
    axis.set_title(f"{column} vs SalePrice")

plt.tight_layout()
plt.show()

## 12. 범주형 변수와 주택가격의 관계

[Question 9]  
- 지역별 주택가격 중앙값에 차이가 있는가?
- 주방 품질과 외장 품질이 높아질수록 가격도 높아지는가?
- 범주별 표본 수가 다를 때 박스플롯을 어떻게 해석해야 하는가?
- 범주형 변수는 향후 회귀모델에 바로 입력할 수 있는가?


### 답변

- 지역별 가격 중앙값은 크게 다르다. 표본 10건 이상 지역 기준으로 `MeadowV` **$88,250**에서 `StoneBr` **$319,000**까지 약 3.6배 차이가 난다. `GrnHill` 2건, `Landmrk` 1건처럼 매우 작은 집단의 중앙값은 일반화하지 않는다.
- `Kitchen_Qual` 중앙값은 Fa **$107,750** → TA **$136,500** → Gd **$201,000** → Ex **$325,624**로 상승한다. `Exter_Qual`도 Fa **$85,000** → TA **$139,000** → Gd **$219,500** → Ex **$370,967**로 뚜렷한 순서가 있다. `Kitchen_Qual=Po`는 1건뿐이라 별도 해석이 어렵다.
- 박스플롯은 중앙값과 분포를 보여주지만 표본 수가 다른 범주의 신뢰도는 같지 않다. 각 범주의 `n`을 함께 표시하고, 표본이 작은 범주는 넓은 불확실성을 고려해야 한다.
- 범주형 변수는 대부분의 회귀모델에 문자열 그대로 입력할 수 없다. `Kitchen_Qual`, `Exter_Qual`은 순서형 인코딩, `Neighborhood`, `House_Style`은 원-핫 인코딩을 우선 검토하며 파이프라인 안에서 처리한다.


In [ ]:
# Neighborhood별 주택가격 중앙값 순서
neighborhood_order = (
    df.groupby("Neighborhood", observed=False)[target]
    .median()
    .sort_values()
    .index
)

plt.figure(figsize=(12, 8))
sns.boxplot(
    data=df,
    y="Neighborhood",
    x=target,
    order=neighborhood_order
)
plt.title("SalePrice by Neighborhood")
plt.tight_layout()
plt.show()

In [ ]:
quality_order = ["Po", "Fa", "TA", "Gd", "Ex"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=df,
    x="Kitchen_Qual",
    y=target,
    order=quality_order,
    ax=axes[0]
)
axes[0].set_title("SalePrice by Kitchen Quality")

sns.boxplot(
    data=df,
    x="Exter_Qual",
    y=target,
    order=quality_order,
    ax=axes[1]
)
axes[1].set_title("SalePrice by Exterior Quality")

plt.tight_layout()
plt.show()

## 13. 주요 변수 교차 탐색

[Question 10]  
- 동일한 생활면적이라도 품질에 따라 가격 차이가 나타나는가?
- Overall_Qual은 면적과 가격의 관계를 어떻게 구분하는가?
- 단일 변수만 보는 것보다 두 변수를 함께 볼 때 새롭게 발견되는 패턴은 무엇인가?


### 답변

- 동일한 `Gr_Liv_Area` 구간에서도 `Overall_Qual`이 높은 점들이 더 높은 `SalePrice` 영역에 위치한다. 즉 면적만 같다고 가격이 같지 않고, 품질이 가격 수준을 위아래로 층화한다.
- `Overall_Qual`은 면적과 가격의 관계에서 일종의 조건/상호작용 변수로 보인다. 품질이 낮은 주택은 면적 증가에 따른 가격 상승 폭이 제한되고, 품질이 높은 주택은 같은 면적에서 더 높은 가격대를 형성한다.
- 두 변수를 함께 보면 단일 변수 분석에서는 보이지 않던 **상호작용, 하위집단, 영향점**을 발견할 수 있다. 특히 생활면적 4,000 ft² 초과 Edwards 3건은 품질 10임에도 가격이 낮아 별도 확인이 필요한 영향점으로 드러난다.
- 다음 모델링에서는 `Gr_Liv_Area × Overall_Qual` 상호작용항이나 트리 기반 비선형 모델을 비교하고, 품질별 잔차 패턴과 극단값 포함/제외 성능을 검증한다.


In [ ]:
plt.figure(figsize=(11, 7))
sns.scatterplot(
    data=df,
    x="Gr_Liv_Area",
    y=target,
    hue="Overall_Qual",
    palette="viridis",
    alpha=0.7
)
plt.title("Living Area, Overall Quality and SalePrice")
plt.tight_layout()
plt.show()

## 14. EDA Findings 작성

[Report]  
아래 질문에 대한 답을 관찰 근거와 함께 작성하세요.

1. 데이터 구조에서 확인한 특징은 무엇인가?
2. 결측치가 있는 변수와 결측의 가능한 의미는 무엇인가?
3. Target 분포의 특징과 향후 고려할 변환은 무엇인가?
4. 이상치 후보가 확인된 변수와 처리 전 확인할 사항은 무엇인가?
5. 주택가격과 관계가 강해 보이는 수치형 변수는 무엇인가?
6. 가격 차이가 뚜렷하게 나타나는 범주형 변수는 무엇인가?
7. 향후 전처리 단계에서 수행해야 할 작업은 무엇인가?

[Example Format]  
- Finding:
- Evidence:
- Possible Action:


### EDA Findings 답변

#### 1. 데이터 구조
- **Finding:** 2,930행 × 15열이며 Target은 `SalePrice`; 설명변수는 수치형 10개, 범주형 4개이다.
- **Evidence:** pandas 자료형 기준 수치형 열은 Target 포함 11개, object형은 4개이다. 연도·등급 변수는 저장 자료형과 분석 의미가 다르다.
- **Possible Action:** 연도는 연식 파생변수, 품질/상태는 순서형 변수로 명시하고 전처리 파이프라인에서 역할별로 분리한다.

#### 2. 결측치와 의미
- **Finding:** 가장 큰 결측은 `Garage_Yr_Blt` 159건(5.43%)이며 대부분 차고가 없다는 구조적 결측이다.
- **Evidence:** 159건 중 157건이 `Garage_Cars=0`; `Total_Bsmt_SF`, `Garage_Cars` 결측은 각 1건이다.
- **Possible Action:** `Has_Garage`를 만들고 ‘차고 없음’과 실제 누락 2건을 분리한다. 소수 결측은 관련 변수와 원자료를 확인한 뒤 0 또는 조건부 중앙값으로 대치한다.

#### 3. Target 분포
- **Finding:** `SalePrice`는 오른쪽 꼬리가 긴 비대칭 분포이다.
- **Evidence:** 평균 $180,796, 중앙값 $160,000, 왜도 1.74; `log1p` 변환 후 왜도는 -0.01이다.
- **Possible Action:** 로그 타깃과 원 타깃 모델을 교차검증하고, 최종 평가는 원 가격 단위 MAE/RMSE도 함께 비교한다.

#### 4. 이상치와 오류
- **Finding:** IQR 후보가 많지만 모두 오류는 아니다. 다만 `Garage_Yr_Blt=2207`, 완전 중복 3행, 고면적·저가격 3건은 우선 검증 대상이다.
- **Evidence:** `Overall_Cond` 252건, `SalePrice` 137건, `Lot_Area` 127건, `Total_Bsmt_SF` 123건이 IQR 경계 밖이다.
- **Possible Action:** 원자료 확인과 영향도 분석 후 처리하며, 극단값을 일괄 삭제하지 않는다. 고면적·저가격 3건은 포함/제외 민감도를 비교한다.

#### 5. 가격과 관계가 강한 수치형 변수
- **Finding:** 품질과 생활면적이 가격을 가장 잘 설명한다.
- **Evidence:** Pearson r은 `Overall_Qual` 0.799, `Gr_Liv_Area` 0.707, `Garage_Cars` 0.648, `Total_Bsmt_SF` 0.632이다.
- **Possible Action:** 품질×면적 상호작용, 비선형성, 다중공선성을 교차검증과 잔차 분석으로 확인한다.

#### 6. 가격 차이가 뚜렷한 범주형 변수
- **Finding:** `Neighborhood`, `Kitchen_Qual`, `Exter_Qual`에서 가격대 차이가 뚜렷하다.
- **Evidence:** 지역 중앙값은 표본 10건 이상 기준 $88,250~$319,000; 주방 품질은 Fa $107,750에서 Ex $325,624, 외장 품질은 Fa $85,000에서 Ex $370,967로 상승한다.
- **Possible Action:** 품질 변수는 순서형, 지역은 원-핫 인코딩하고 희소 범주는 학습 데이터 기준으로 통합한다.

#### 7. 향후 전처리
- **Finding:** 전처리의 핵심은 의미 보존과 데이터 누수 방지이다.
- **Evidence:** 구조적 결측, 강한 왜도, 오류 연도, 중복, 희소 범주, 서로 다른 단위가 동시에 존재한다.
- **Possible Action:** ① 결측 의미 분리 및 indicator 생성 → ② 오류·중복 검증 → ③ 로그 변환 → ④ 순서형/원-핫 인코딩과 희소 범주 처리 → ⑤ 필요한 모델에 스케일링을 적용한다. 모든 변환은 `Pipeline`/`ColumnTransformer` 안에서 학습 세트에만 fit한다.
